# 12.1 Browsing through results: the display command

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

The easiest way to examine `data` and result values is to type `display` and a description
of what you want to look at. The `display` command automatically formats the values
in an intuitive and familiar arrangement; as much as possible, it uses the same list and
`table` formats as the `data` statements described in [Chapter 9](../09/09.md). Our examples use parameters
and variables from models defined in other chapters.

As we will describe in more detail in [Section 12.7](./12_7_general_output_commands.ipynb), it is possible to capture the output
of `display` commands in a file, by adding >filename to the end of a `display` command;
this redirection mechanism applies as well to most other commands that produce
output.

Displaying sets
The contents of sets are shown by typing `display` and a list of set names. This
example is taken from the `model` of [Figure 6-2a](../06/6_2_subsets_and_slices_of_ordered_pairs.ipynb#fig-6-2a):

```ampl
ampl: display ORIG, DEST, LINKS;
set ORIG := GARY CLEV PITT;
set DEST := FRA DET LAN WIN STL FRE LAF;
set LINKS :=
       (GARY,DET)   (GARY,LAF)   (CLEV,LAN)   (CLEV,LAF)   (PITT,STL)
       (GARY,LAN)   (CLEV,FRA)   (CLEV,WIN)   (PITT,FRA)   (PITT,FRE)
       (GARY,STL)   (CLEV,DET)   (CLEV,STL)   (PITT,WIN);
```

If you specify the name of an indexed collection of sets, each set in the collection is
shown (from [Figure 6-3](../06/6_5_indexed_collections_of_sets.ipynb#fig-6-3)):

```ampl
ampl: display PROD, AREA;
set PROD := bands coils;
set AREA[bands] := east north;
set AREA[coils] := east west export;
```

Particular members of an indexed collection can be viewed by subscripting, as in
`display` AREA["bands"].

The argument of `display` need not be a declared set; it can be any of the expressions
described in [Chapter 5](../05/05.md) or 6 that evaluate to sets. For example, you can show the
union of all the sets AREA[p]:

```ampl
ampl: display union {p in PROD} AREA[p];
set union {p in PROD} AREA[p] := east north west export;
```

or the set of all transportation links on which the shipping cost is greater than 500:

```ampl
ampl: display {(i,j) in LINKS: cost[i,j] * Trans[i,j] > 500};
set {(i,j) in LINKS: cost[i,j]*Trans[i,j] > 500} :=
(GARY,STL)   (CLEV,DET)   (CLEV,WIN)   (PITT,FRA)   (PITT,FRE)
(GARY,LAF)   (CLEV,LAN)   (CLEV,LAF)   (PITT,STL);
```

Because the membership of this set depends upon the current values of the variables
Trans[i,j], you could not refer to it in a `model`, but it is legal in a `display` command,
where variables are treated the same as parameters.

Displaying parameters and variables
The `display` command can show the value of a scalar `model` component:

```ampl
ampl: display T;
T = 4
```

or the values of individual components from an indexed collection ([Figure 1-6b](../01/tut_1_6.ipynb#fig-1-6)):

```ampl
ampl: display avail["reheat"], avail["roll"];
avail['reheat'] = 35
avail['roll'] = 40
```

or an arbitrary expression:

```ampl
ampl: display sin(1)ˆ2 + cos(1)ˆ2;
sin(1)ˆ2 + cos(1)ˆ2 = 1
```

The major use of `display`, however, is to show whole indexed collections of `data`. For
"one-dimensional" `data` — parameters or variables indexed over a simple set — AMPL
uses a column format ([Figure 4-6b](../04/4_3_a_model_of_production_and_transportation.ipynb#fig-4-6)):

```ampl
ampl: display avail;
avail [*] :=
reheat 35
       roll 40
;
```

For "two-dimensional" parameters or variables — indexed over a set of pairs or two
simple sets — AMPL forms a list for small amounts of `data` ([Figure 4-1](../04/4_1_a_multicommodity_transportation_model.ipynb#fig-4-1)):

```ampl
ampl: display supply;
supply :=
CLEV bands    700
CLEV coils   1600
CLEV plate    300
GARY bands    400
GARY coils    800
GARY plate    200
PITT bands    800
PITT coils   1800
PITT plate    300
;
```

or a `table` for larger amounts:

```ampl
ampl: display demand;
demand [*,*]
:   bands coils plate  :=
DET   300    750  100
FRA   300    500  100
FRE   225    850  100
LAF   250    500  250
LAN   100    400    0
STL   650    950  200
WIN    75    250   50
;
```

You can control the choice between formats by setting option display_1col, which is
described in the next section.

A parameter or variable (or any other `model` entity) indexed over a set of ordered pairs
is also considered to be a two-dimensional object and is displayed in a similar manner.
Here is the `display` for a parameter indexed over the set LINKS that was displayed earlier
in this section (from [Figure 6-2a](../06/6_2_subsets_and_slices_of_ordered_pairs.ipynb#fig-6-2a)):

```ampl
ampl: display cost;
cost :=
CLEV DET    9
CLEV FRA   27
CLEV LAF   17
CLEV LAN   12
CLEV STL   26
CLEV WIN    9
GARY DET   14
GARY LAF    8
GARY LAN   11
GARY STL   16
PITT FRA   24
PITT FRE   99
PITT STL   28
PITT WIN   13
;
```

This, too, can be made to appear in a `table` format, as the next section will show.

To `display` values indexed in three or more dimensions, AMPL again forms lists for
small amounts of `data`. Multi-dimensional entities more often involve `data` in large
quantities, however, in which case AMPL "slices" the values into two-dimensional tables,
as in the case of this variable from [Figure 4-6](../04/4_3_a_model_of_production_and_transportation.ipynb#fig-4-6):

```ampl
ampl: display Trans;
Trans [CLEV,*,*]
:   bands coils plate             :=
DET    0    750    0
FRA    0      0    0
FRE    0      0    0
LAF    0    500    0
LAN    0    400    0
STL    0     50    0
WIN    0    250    0
  [GARY,*,*]
:    bands coils plate            :=
DET      0     0     0
FRA      0     0     0
FRE    225   850   100
LAF    250     0     0
LAN      0     0     0
STL    650   900   200
WIN      0     0     0
  [PITT,*,*]
:    bands coils plate            :=
DET    300     0   100
FRA    300   500   100
FRE      0     0     0
LAF      0     0   250
LAN    100     0     0
STL      0     0     0
WIN     75     0    50
;
```

At the head of the first `table`, the template [CLEV,*,*] indicates that the slice is
through CLEV in the first component, so the entry in row LAF and column coils says
that Trans["CLEV","LAF","coils"] is 500. Since the first index of Trans is
always CLEV, GARY or PITT in this case, there are three slice tables in all. But AMPL
does not always slice through the first component; it picks the slices so that the `display`
will contain the fewest possible tables.

A `display` of two or more components of the same dimensionality is always presented
in a list format, whether the components are one-dimensional ([Figure 4-4](../04/4_3_a_model_of_production_and_transportation.ipynb#fig-4-4)):

```ampl
ampl: display inv0, prodcost, invcost;
:     inv0 prodcost invcost    :=
bands   10     10      2.5
coils    0     11      3
;
```

or two-dimensional ([Figure 4-6](../04/4_3_a_model_of_production_and_transportation.ipynb#fig-4-6)):

```ampl
ampl: display rate, make_cost, Make;
:           rate make_cost   Make    :=
CLEV bands   190     190        0
CLEV coils   130     170     1950
CLEV plate   160     185        0
GARY bands   200     180     1125
GARY coils   140     170     1750
GARY plate   160     180      300
PITT bands   230     190      775
PITT coils   160     180      500
PITT plate   170     185      500
;
```

or any higher dimension. The indices appear in a list to the left, with the last one changing
most rapidly.

As you can see from these examples, `display` normally arranges row and column
labels in alphabetical or numerical order, regardless of the order in which they might have
been given in your `data` file. When the labels come from an ordered set, however, the
original ordering is honored ([Figure 5-3](../05/5_6_ordered_sets.ipynb#fig-5-3)):

```ampl
ampl: display avail;
avail [*] :=
27sep 40
04oct 40
11oct 32
18oct 40
;
```

For this reason, it can be worthwhile to declare certain sets of your `model` to be ordered,
even if their ordering plays no explicit role in your formulation.

#### Displaying indexed expressions

The `display` command can show the value of any arithmetic expression that is
valid in an AMPL `model`. Single-valued expressions pose no difficulty, as in the case of
these three profit components from [Figure 4-4](../04/4_3_a_model_of_production_and_transportation.ipynb#fig-4-4):

```ampl
ampl:   display sum {p in PROD,t in 1..T} revenue[p,t]*Sell[p,t],
ampl?           sum {p in PROD,t in 1..T} prodcost[p]*Make[p,t],
ampl?           sum {p in PROD,t in 1..T} invcost[p]*Inv[p,t];
sum{p   in PROD, t in 1 .. T} revenue[p,t]*Sell[p,t] = 787810
sum{p   in PROD, t in 1 .. T} prodcost[p]*Make[p,t] = 269477
sum{p   in PROD, t in 1 .. T} invcost[p]*Inv[p,t] = 3300
```

Suppose, however, that you want to see all the individual values of
`revenue[p,t] * Sell[p,t]`. Since you can type `display` revenue, Sell to
`display` the separate values of revenue[p,t] and Sell[p,t], you might want
to ask for the products of these values by typing:

```ampl
ampl: display revenue * Sell;
syntax error
context: display revenue >>> * <<< Sell;
```

AMPL does not recognize this kind of array arithmetic. To `display` an indexed collection
of expressions, you must specify the indexing explicitly:

```ampl
ampl: display {p in PROD, t in 1..T} revenue[p,t]*Sell[p,t];
revenue[p,t]*Sell[p,t] [*,*] (tr)
:   bands    coils     :=
1   150000     9210
2   156000    87500
3    37800   129500
4    54000   163800
;
```

To apply the same indexing to two or more expressions, enclose a list of them in parentheses
after the indexing expression:

```ampl
ampl:   display {p in PROD, t in 1..T}
ampl?      (revenue[p,t]*Sell[p,t], prodcost[p]*Make[p,t]);
:         revenue[p,t]*Sell[p,t] prodcost[p]*Make[p,t]    :=
bands   1          150000                 59900
bands   2          156000                 60000
bands   3           37800                 14000
bands   4           54000                 20000
coils   1            9210                 15477
coils   2           87500                 15400
coils   3          129500                 38500
coils   4          163800                 46200
;
```

An indexing expression followed by an expression or parenthesized list of expressions is
treated as a single `display` item, which specifies some indexed collection of values. A
`display` command may contain one of these items as above, or a comma-separated list
of them.

The presentation of the values for indexed expressions follows the same rules as for
individual parameters and variables. In fact, you can regard a command like

```ampl
display revenue, Sell
```

as shorthand for

```ampl
ampl:   display {p in PROD, t in 1..T} (revenue[p,t],Sell[p,t]);
:         revenue[p,t] Sell[p,t]    :=
bands   1       25        6000
bands   2       26        6000
bands   3       27        1400
bands   4       27        2000
coils   1       30         307
coils   2       35        2500
coils   3       37        3500
coils   4       39        4200
;
```

If you rearrange the indexing expression so that `t` in 1..T comes first, however, the
rows of the list are instead sorted first on the members of 1..T:

```ampl
ampl: display {t in 1..T, p in PROD} (revenue[p,t],Sell[p,t]);
:       revenue[p,t] Sell[p,t]    :=
1 bands       25        6000
1 coils       30         307
2 bands       26        6000
2 coils       35        2500
3 bands       27        1400
3 coils       37        3500
4 bands       27        2000
4 coils       39        4200
;
```

This change in the default presentation can only be achieved by placing an explicit indexing
expression after `display`.
In addition to indexing individual `display` items, you can specify a set over which the
whole `display` command is indexed — that is, you can ask that the command be executed
once for each member of an indexing set. This feature is particularly useful for
rearranging slices of multidimensional tables. When, earlier in this section, we displayed
the three-dimensional variable Trans indexed over {ORIG,DEST,PROD}, AMPL
chose to slice the values through members of ORIG to produce a series of twodimensional
tables.

What if you want to `display` slices through PROD? Rearranging the indexing expression,
as in our previous example, will not reliably have the desired effect; the `display`
command always picks the smallest indexing set, and where there is more than one that is
smallest, it does not necessarily choose the first. Instead, you can say explicitly that you
want a separate `display` for each `p` in PROD:

```ampl
ampl: display {p in PROD}:
ampl?    {i in ORIG, j in DEST} Trans[i,j,p];
Trans[i,j,'bands'] [*,*] (tr)
:   CLEV GARY PITT      :=
DET   0      0   300
FRA   0      0   300
FRE   0    225     0
LAF   0    250     0
LAN   0      0   100
STL   0    650     0
WIN   0      0    75
;

Trans[i,j,'coils'] [*,*] (tr)
:    CLEV GARY PITT      :=
DET   750     0     0
FRA     0     0   500
FRE     0   850     0
LAF   500     0     0
LAN   400     0     0
STL    50   900     0
WIN   250     0     0
;

Trans[i,j,'plate'] [*,*] (tr)
:   CLEV GARY PITT      :=
DET   0      0   100
FRA   0      0   100
FRE   0    100     0
LAF   0      0   250
LAN   0      0     0
STL   0    200     0
WIN   0      0    50
;
```

As this example shows, if a `display` command specifies an indexing expression right at
the beginning, followed by a colon, the indexing set applies to the whole command. For

```ampl
display_1col        maximum elements for a table to be displayed in list format (20)
display_transpose   transpose tables if rows - columns < display_transpose (0)
display_width       maximum line width (79)
gutter_width        separation between table columns (3)
omit_zero_cols      if not 0, omit all-zero columns from displays (0)
omit_zero_rows      if not 0, omit all-zero rows from displays (0)
```

**[Table 12-1](./12_2_formatting_options_for_display.ipynb#`table`-12-1):** Formatting options for `display` (with default values).

each member of the set, the expressions following the colon are evaluated and displayed
separately.